# SFT lora: Medgemma 27B text it

In [1]:
import os

os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_SILENT"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print("VRAM GB:", round(props.total_memory / 1024**3, 1))

CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM GB: 95.0


In [3]:
from datasets import load_dataset

In [4]:
from finetune_medgemma_lora import (
    train_lora,
    validate_jsonl,
    summarize_token_lengths,
    preview_training_pair,
    evaluate_jsonl,
    load_model_for_inference,
    generate_extraction,
)

# Prepre dataset
{"report":"Compte rendu anatomopathologique ...","label":{"echantillon":"tumorectomie","numero_gang_prel":14,"numero_gang_att":2,"taille":18,"diagnostic":"carcinome canalaire infiltrant","grade":"SBR2","emboles":"non","ki67":22,"re":"positifs","rp":"positifs","her2":"négatif","marges_saines":"oui","rupture_capsulaire":"non"}}

Categorical missing/ambiguous values -> "unknown"

Numeric missing/ambiguous values -> null

In [5]:
import ast

def flatten_entry_data(entry_data):
    if isinstance(entry_data, str):
        entry_data = ast.literal_eval(entry_data)

    return {
        att: info.get("value", "unknown")
        for att, info in entry_data.items()
    }

    
def normalize_examples_dataframe(data):
    """Return a dataframe with at least columns: entry_data, text.

    Accepted inputs:
    - pd.DataFrame with entry_data/text columns
    - pd.DataFrame with eval column containing list(s) of records
    - dict with key "eval" containing records
    - list of records
    """
    if isinstance(data, dict) and "eval" in data:
        return pd.DataFrame(data["eval"])

    if isinstance(data, list):
        return pd.DataFrame(data)

    if not isinstance(data, pd.DataFrame):
        raise TypeError("data must be a DataFrame, dict with 'eval', or list of records")

    if {"entry_data", "text"}.issubset(data.columns):
        return data.copy()

    if "eval" in data.columns:
        records = []
        for value in data["eval"].dropna():
            if isinstance(value, list):
                records.extend(value)
            elif isinstance(value, dict):
                records.append(value)
            else:
                raise TypeError(f"Unexpected eval cell type: {type(value)}")
        return pd.DataFrame(records)

    raise ValueError("Could not find columns entry_data/text or an eval column.")





In [6]:
import pandas as pd
import json

OUTPUTFOLDER='training_data/ouputddkd1_2'

df_rk_1_2=pd.read_csv("../LLMEvaluation/IE/rk1_2_selected.csv")

normalized_df_rk_1_2 = normalize_examples_dataframe(df_rk_1_2)
normalized_df_rk_1_2["entry_data"] = normalized_df_rk_1_2["entry_data"].apply(flatten_entry_data)

normalized_df_rk_1_2.columns

Index(['index', 'text', 'entry_data', 'source'], dtype='object')

In [7]:
import pandas as pd
import json
import ast

def parse_entry_data(x):
    if isinstance(x, dict):
        return x

    if pd.isna(x):
        return {}

    if isinstance(x, str):
        try:
            return json.loads(x)
        except json.JSONDecodeError:
            return ast.literal_eval(x)

    return {}

In [8]:
normalized_df_rk_1_2["entry_data_parsed"] = normalized_df_rk_1_2["entry_data"].apply(parse_entry_data)

entry_df = pd.json_normalize(normalized_df_rk_1_2["entry_data_parsed"])
entry_df = entry_df.drop(
    columns=["re_perc", "rp_perc"],
    errors="ignore"
)

COLUMN_MAPPING = {
    "type_diagnostique": "echantillon",
    "ganglions_preleves": "numero_gang_prel",
    "ganglions_atteints": "numero_gang_att",
    "taille_tumor_0": "taille",
    "ref_morpho_name_tumor_0": "diagnostic",
    "ref_grade_tumor_0": "grade",
    "embols_vasculaires_tumor_0": "emboles",
    "ki67_tumor_0": "ki67",
    "re_tumor_0": "re",
    "rp_tumor_0": "rp",
    "cerb_tumor_0": "her2",
    "marges_saines_tumor_0": "marges_saines",
    "rupture_capsulaire": "rupture_capsulaire",
}


entry_df = entry_df.rename(columns=COLUMN_MAPPING)

entry_df.columns

Index(['echantillon', 'numero_gang_prel', 'numero_gang_att',
       'rupture_capsulaire', 'taille', 're', 'rp', 'emboles', 'her2',
       'marges_saines', 'ki67', 'diagnostic', 'grade'],
      dtype='object')

In [1]:
FIELD_ORDER = [
    "echantillon",
    "numero_gang_prel",
    "numero_gang_att",
    "taille",
    "diagnostic",
    "grade",
    "emboles",
    "ki67",
    "re",
    "rp",
    "her2",
    "marges_saines",
    "rupture_capsulaire",
]

entry_df = entry_df.reindex(columns=FIELD_ORDER)
entry_df.head()

In [2]:
flat_df = pd.concat(
    [
        normalized_df_rk_1_2[['index',"text"]].reset_index(drop=True),
        entry_df.reset_index(drop=True),
    ],
    axis=1,
)

flat_df.head()

In [13]:
NUMERIC_FIELDS = {
    "numero_gang_prel",
    "numero_gang_att",
    "taille",
    "ki67",
}

CATEGORICAL_FIELDS = set(FIELD_ORDER) - NUMERIC_FIELDS

def clean_numeric(value):
    if value is None or pd.isna(value) or value == "":
        return None

    if isinstance(value, str):
        value = value.strip()
        value = value.replace("%", "")
        value = value.replace(",", ".")
        if value == "":
            return None

    try:
        return int(float(value))
    except Exception:
        return None


def clean_categorical(value):
    if value is None or pd.isna(value) or value == "":
        return "unknown"

    value = str(value).strip()

    if value.lower() in ["nan", "none", "null", "na", "n/a"]:
        return "unknown"

    return value


def clean_label_row(row):
    label = {}

    for field in FIELD_ORDER:
        value = row.get(field)

        if field in NUMERIC_FIELDS:
            label[field] = clean_numeric(value)
        else:
            label[field] = clean_categorical(value)

    return label

# load data

In [14]:


records = []


for i, row in flat_df.iterrows():
    label = clean_label_row(entry_df.iloc[i])

    records.append({
        "report": str(row["text"]).strip(),
        "label": label,
    })

with open(OUTPUTFOLDER+"/all_data.jsonl", "w", encoding="utf-8") as f:
    for record in records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


with open(OUTPUTFOLDER+"/all_data.jsonl", "r", encoding="utf-8") as f:
    example = json.loads(next(f))

print(json.dumps(example, indent=2, ensure_ascii=False))

{
  "report": "## COMPTE RENDU ANATOMOPATHOLOGIQUE\n\n**Titre :** MICROBIOPSIES MAMMAIRES : ADÉNOCARCINOME CANALAIRE INFILTRANT\n\n**Examen macroscopique :**\nFragments de biopsies mammaires mesurant ensemble 1,4 cm dans leur plus grande dimension. Les prélèvements sont entièrement inclus.\n\n**Examen microscopique :**\nOn observe une prolifération néoplasique maligne mammaire primitive faite de massifs et de travées infiltrants. Les atypies sont modérées. Les mitoses sont peu nombreuses. Le stroma est fibreux desmoplastique.\n\n**Étude immunohistochimique :**\n*   Récepteurs hormonaux :\n    *   Œstrogènes (RE) : négatifs.\n    *   Progestérone (RP) : positifs ; 90 % de cellules exprimées.\n*   HER2 (CerbB2) : 0.\n*   Prolifération cellulaire (Ki67) : non évaluée.\n\n**Conclusion :**\nAdénocarcinome mammaire infiltrant de type canalaire évalué sur biopsies à un grade II SBR modifié (SBR2).\nStatut des récepteurs hormonaux : RE négatif, RP positif (90%).\nStatut Her2Neu : 0.\nNombre de

### split train_dev_val

In [15]:
import random

random.seed(42)

with open(OUTPUTFOLDER+"/all_data.jsonl", "r", encoding="utf-8") as f:
    items = [json.loads(line) for line in f]

random.shuffle(items)

n = len(items)
n_train = int(0.9 * n)
n_valid = int(0.1 * n)

train_items = items[:n_train]
valid_items = items[n_train:n_train+n_valid]

def write_jsonl(path, data):
    with open(path, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

write_jsonl(OUTPUTFOLDER+"/train.jsonl", train_items)
write_jsonl(OUTPUTFOLDER+"/valid.jsonl", valid_items)


print(len(train_items), len(valid_items))

666 74


# finetune

In [3]:
from finetune_medgemma_lora import summarize_token_lengths

MODEL_PATH= "google/medgemma-27b-text-it"

summarize_token_lengths(OUTPUTFOLDER+"/train.jsonl",model_id=MODEL_PATH)

In [4]:
from finetune_medgemma_lora import train_lora

trainer, tokenizer = train_lora(
    train_file=OUTPUTFOLDER+"/train.jsonl",
    valid_file=OUTPUTFOLDER+"/valid.jsonl",
    output_dir="models/medgemma-pathology-full-lora-1_2OUTSEL",

    model_id=MODEL_PATH,

    max_seq_length=2048,
    num_train_epochs=2,
    learning_rate=1e-4,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,

    lora_r=32,
    lora_alpha=64,
    lora_dropout=0.05,

    eval_steps=25,
    save_steps=25,
    save_total_limit=3,

    bf16=True,
    fp16=False,
)